In [ ]:
import os
import datetime
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import numpy as np
import polars as pl

# Project imports
from utils.config import Algorithm, DatasetName, ModelName, Metric
from utils.data_utils import get_dataloaders
from utils.train_utils import train_and_evaluate
from utils.plot_utils import plot_fitness_evolution

from algorithms.random_search import random_search
from algorithms.local_search import local_search
from algorithms.genetic import genetic_algorithm
from algorithms.memetic import memetic_algorithm

def set_seed(seed: int = 42):
    """Ensures completely reproducible experiments across all frameworks."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def setup_experiment_directories(exp_name: str):
    """Creates directory structures for logs, intermediate plots, and persistent CSV targets."""
    os.makedirs(f"img/{exp_name}", exist_ok=True)
    os.makedirs(f"logs/{exp_name}", exist_ok=True)
    os.makedirs(f"results/csvs", exist_ok=True)
    os.makedirs(f"tmp", exist_ok=True)

# =====================================================================
# EXPERIMENT CONFIGURATION (INTERACTIVE CELL PANEL)
# =====================================================================
TARGET_ALGORITHM = Algorithm.GENETIC       # Choose your Metaheuristic
TARGET_DATASET = DatasetName.MNIST         # Choose your Target Dataset
TARGET_MODEL = ModelName.ALEXNET           # Choose target Network Architecture
TARGET_METRIC = Metric.ACCURACY            # Metric targeted by fitness_wrapper
ADJUST_SIZE = False                        # True: Database sizing fluctuates (FREE) | False: Strict keep_percentage

KEEP_PERCENTAGE = 0.25                     # Percentage base bounds constraint
MAX_EVALUATIONS = 50                       # Maximum optimization computation limit budget
PATIENCE = 15                              # Early stopping threshold for stagnating evaluations
EPOCHS_PER_EVAL = 5                        # Epoch count per individual fitness evaluation train cycle

# Automatically generate a descriptive, serialized directory key
SIZE_TAG = "free" if ADJUST_SIZE else f"fixed_{int(KEEP_PERCENTAGE*100)}pct"
EXPERIMENT_NAME = f"{TARGET_ALGORITHM.value}_{TARGET_DATASET.value}_{TARGET_MODEL.value}_{SIZE_TAG}"

print(f"CUDA status check: {torch.cuda.is_available()}")
print(f"Initializing optimization stack for context: {EXPERIMENT_NAME}")

# --- Automatic retrieval of the base training dataset size ---
temp_loaders, _ = get_dataloaders(
    dataset_name=TARGET_DATASET.value, 
    data_dir="./data", 
    selection_mask=None, 
    train_split=0.8
)
TOTAL_INSTANCES = len(temp_loaders["train"].dataset) 
print(f"Detected training base structure: {TOTAL_INSTANCES} training instances.")

def get_model(model_enum: ModelName, num_classes: int) -> nn.Module:
    """Instantiates target pre-trained architecture, freezes backbones, and re-allocates classification heads."""
    if model_enum == ModelName.ALEXNET:
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        for param in model.parameters():
            param.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        
    elif model_enum == ModelName.RESNEXT:
        model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT)
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Model framework blueprint '{model_enum}' not found.")
    return model

def fitness_wrapper(mask: dict) -> dict:
    """Bridges search metaheuristics with PyTorch Deep Learning execution loops."""
    loaders, classes = get_dataloaders(
        dataset_name=TARGET_DATASET.value,
        data_dir="./data",
        selection_mask=mask,
        batch_size=32,
        train_split=0.8
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model(TARGET_MODEL, len(classes)).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    metrics = train_and_evaluate(
        model=model,
        loaders=loaders,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=EPOCHS_PER_EVAL,
        task_id=EXPERIMENT_NAME
    )
    
    del model
    torch.cuda.empty_cache()
    return metrics

# Persistence path configurations
setup_experiment_directories(EXPERIMENT_NAME)
global_csv_path = "results/csvs/global_experiments_log.csv"
history_csv_path = "results/csvs/history_evaluations_log.csv"

# Rigorous statistical validation seeds array
semillas = [42, 123, 456, 789, 1024]

for current_seed in semillas:
    print(f"\n\n{'='*70}")
    print(f"LAUNCHING SEED ITERATION WORKFLOW: {current_seed}")
    print(f"{'='*70}")
    
    set_seed(current_seed)
    start_time = datetime.datetime.now()
    
    # 1. EVOLUTIONARY RE-ALLOCATION MATRIX OPTIMIZATION
    if TARGET_ALGORITHM == Algorithm.RANDOM_SEARCH:
        best_mask, best_fitness, fitness_history, best_fitness_history, evals_done = random_search(
            fitness_func=fitness_wrapper, total_instances=TOTAL_INSTANCES,
            keep_percentage=KEEP_PERCENTAGE, max_evaluations=MAX_EVALUATIONS,
            patience=PATIENCE, target_metric=TARGET_METRIC
        )
    elif TARGET_ALGORITHM == Algorithm.LOCAL_SEARCH:
        best_mask, best_fitness, fitness_history, best_fitness_history, evals_done = local_search(
            fitness_func=fitness_wrapper, total_instances=TOTAL_INSTANCES,
            keep_percentage=KEEP_PERCENTAGE, max_evaluations=MAX_EVALUATIONS,
            patience=PATIENCE, neighbor_size=10, target_metric=TARGET_METRIC,
            adjust_size=ADJUST_SIZE
        )
    elif TARGET_ALGORITHM == Algorithm.GENETIC:
        best_mask, best_fitness, fitness_history, best_fitness_history, evals_done = genetic_algorithm(
            fitness_func=fitness_wrapper, total_instances=TOTAL_INSTANCES,
            population_size=10, keep_percentage=KEEP_PERCENTAGE, max_evaluations=MAX_EVALUATIONS,
            patience=PATIENCE, tournament_size=3, mutation_rate=0.1, target_metric=TARGET_METRIC,
            adjust_size=ADJUST_SIZE
        )
    elif TARGET_ALGORITHM == Algorithm.MEMETIC:
        best_mask, best_fitness, fitness_history, best_fitness_history, evals_done = memetic_algorithm(
            fitness_func=fitness_wrapper, total_instances=TOTAL_INSTANCES,
            population_size=10, keep_percentage=KEEP_PERCENTAGE, max_evaluations=MAX_EVALUATIONS,
            patience=PATIENCE, tournament_size=3, mutation_rate=0.1, 
            local_search_probability=0.2, local_search_evaluations=10, local_search_neighbor_size=5,
            target_metric=TARGET_METRIC, adjust_size=ADJUST_SIZE
        )
    else:
        raise ValueError(f"Algorithm workflow block mapping for '{TARGET_ALGORITHM.value}' is absent.")

    end_time = datetime.datetime.now()
    duration_str = str(datetime.timedelta(seconds=int((end_time - start_time).total_seconds())))

    if best_fitness > 0.0:
        # Save evolution plots uniquely identified by seed
        img_dir = f"img/{EXPERIMENT_NAME}"
        plot_fitness_evolution(
            fitness_history=best_fitness_history,
            initial_percentage=int(KEEP_PERCENTAGE * 100),
            algorithm_name=f"{TARGET_ALGORITHM.value}_seed_{current_seed}",
            metric=TARGET_METRIC.value,
            model=TARGET_MODEL.value,
            output_dir=img_dir
        )

        # 2. GRANULAR PERSISTENCE: Save ALL individual intermediate search steps
        print(f"\nStoring {len(fitness_history)} search evaluations into step-by-step history logs...")
        cleaned_history_entries = []
        for raw_entry in fitness_history:
            entry = {k.capitalize() if k not in ["Algorithm", "Initial Percentage", "Final Percentage", "Iteration"] else k: v 
                     for k, v in raw_entry.items()}
            # Enforce experiment configuration contextual keys
            entry.update({
                "Experiment_Name": EXPERIMENT_NAME,
                "Dataset": TARGET_DATASET.value,
                "Model": TARGET_MODEL.value,
                "Seed": current_seed,
                "Algorithm": f"{TARGET_ALGORITHM.value}_FREE" if ADJUST_SIZE else TARGET_ALGORITHM.value
            })
            cleaned_history_entries.append(entry)
            
        df_history_run = pl.DataFrame(cleaned_history_entries)
        if os.path.exists(history_csv_path):
            df_history_existing = pl.read_csv(history_csv_path)
            df_history_final = pl.concat([df_history_existing, df_history_run], how="diagonal")
        else:
            df_history_final = df_history_run
        df_history_final.write_csv(history_csv_path)

        # 3. CRITICAL EVALUATION PHASE AGAINST HOLDOUT TEST SET
        print("\n--- Deploying Final Generalization Performance Evaluation Against HOLDOUT TEST Set ---")
        final_loaders, final_classes = get_dataloaders(
            dataset_name=TARGET_DATASET.value,
            data_dir="./data",
            selection_mask=best_mask,
            batch_size=32,
            train_split=0.8
        )
        
        final_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        final_model = get_model(TARGET_MODEL, len(final_classes)).to(final_device)
        
        final_criterion = nn.CrossEntropyLoss()
        final_optimizer = optim.Adam(final_model.parameters(), lr=0.001)
        
        final_metrics = train_and_evaluate(
            model=final_model, 
            loaders=final_loaders, 
            criterion=final_criterion, 
            optimizer=final_optimizer, 
            device=final_device, 
            num_epochs=EPOCHS_PER_EVAL, 
            task_id=f"{EXPERIMENT_NAME}_final_seed_{current_seed}",
            eval_split="test"
        )
        
        del final_model
        torch.cuda.empty_cache()

        # 4. FINAL GLOBAL SUMMARY PERSISTENCE RECORDING
        final_pct = sum(best_mask.values()) / TOTAL_INSTANCES
        results_record = {
            "Experiment_Name": EXPERIMENT_NAME,
            "Algorithm": TARGET_ALGORITHM.value,
            "Dataset": TARGET_DATASET.value,
            "Model": TARGET_MODEL.value,
            "Adjust_Size_Flag": str(ADJUST_SIZE),
            "Initial_Percentage": KEEP_PERCENTAGE,
            "Final_Selected_Percentage": final_pct,
            "Seed": current_seed,
            "Duration": duration_str,
            "Evaluations_Done": evals_done
        }
        results_record.update({f"Test_{k.capitalize()}": v for k, v in final_metrics.items()})

        df_current_run = pl.DataFrame({k: [v] for k, v in results_record.items()})
        
        if os.path.exists(global_csv_path):
            df_existing = pl.read_csv(global_csv_path)
            df_final = pl.concat([df_existing, df_current_run], how="diagonal")
        else:
            df_final = df_current_run
            
        df_final.write_csv(global_csv_path)
        print(f"Global consolidated summaries updated for seed {current_seed}.")
        display(df_current_run)
    else:
        print(f"Execution Error: Seed {current_seed} failed to compute optimization space metrics safely.")

print(f"\n{'='*70}\n[SUCCESS] ALL REQUESTED EXPERIMENTAL BLOCKS COMPUTED PERFECTLY\n{'='*70}")

CUDA status check: True
Initializing optimization stack for context: GEN_MNIST_alexnet_fixed_25pct
[MNIST] Images loaded. Active Training Instances: 48000/48000
Detected training base structure: 48000 training instances.


LAUNCHING SEED ITERATION WORKFLOW: 42
Starting GEN (Target: ACCURACY | Initial Retention: 25.0%)
--- [GEN] Initial Pop Evaluation 1/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 2/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 3/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 4/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 5/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 6/10 ---
[MNIST] Images loaded. Active Training Instances: 12000/48000
--- [GEN] Initial Pop Evaluation 7/10 ---
[MNIST] Images